# envs

> Helpers for fast, repeatable Python environments on Colab runtimes

- skip_exec: true

Consumer Colab runtimes boot Google's fixed base image rather than a custom Docker container. That image includes CUDA, PyTorch, JAX, `uv`, and `gsutil`. Small extras install in a second with `uv`. Larger or version-pinned environments work better with lockfiles, archived virtual environments, or warm runtime reuse. Below we add helpers on `ColabRuntime` for each approach and run them against a live CPU runtime.

In [ ]:
#| default_exp envs

In [ ]:
#| export
import subprocess
from pathlib import Path
from fastcore.utils import *
from fastcolab.core import *

In [ ]:
#| hide
import json, os, time, uuid
from nbdev.showdoc import *
from fastcore.test import *
from fastcolab.core import _drive_folder_id

## Starting a live runtime and checking the base image

Connect to Colab with `Colab.connect` and start a single High-RAM CPU runtime that we will reuse across every section below.

In [ ]:
acct = os.environ['FASTCOLAB_ACCOUNT']
c = await Colab.connect(account=acct)
rt = await c.start('cpu')
rt

Check what tools and packages Google's base image already provides. Seeing `uv` and `tar` present (and `zstd` absent) sets the defaults for the helpers below.

In [ ]:
probe = await rt.exec('''
import shutil, subprocess, sys
def ver(cmd):
    p = shutil.which(cmd)
    if not p: return None
    return subprocess.run([p, '--version'], capture_output=True, text=True).stdout.strip().splitlines()[0]
for cmd in ('uv', 'tar', 'zstd', 'gsutil'): print(f'{cmd}: {ver(cmd)}')
print('python:', sys.version.split()[0], sys.executable)
for pkg in ('torch', 'chevron', 'shortuuid'):
    try: __import__(pkg); status = 'installed'
    except ImportError: status = 'missing'
    print(f'{pkg}: {status}')
''')
probe_txt = outs_text(probe)
assert 'uv:' in probe_txt and 'chevron: missing' in probe_txt and 'shortuuid: missing' in probe_txt
print(probe_txt.strip())

The environment helpers run `uv` and `tar` on the remote runtime. `_run_cmd` invokes `subprocess.run` on the kernel and prints stdout and stderr. Any non-zero exit code raises `RuntimeError`.

In [ ]:
#| export
async def _run_cmd(
    rt:ColabRuntime, # Active Colab runtime
    args:list|str, # Command argv list (preferred) or shell command string
    timeout:float=None, # Execution timeout in seconds
)->str:
    "Run a command on `rt` and return combined output, raising `RuntimeError` on non-zero exit."
    shell = isinstance(args, str)
    code = f'''
import subprocess
_r = subprocess.run({args!r}, shell={shell!r}, capture_output=True, text=True)
if _r.stdout: print(_r.stdout, end='')
if _r.stderr: print(_r.stderr, end='')
if _r.returncode: raise RuntimeError(f'exit {{_r.returncode}}')
'''
    outs = await rt.exec(code, timeout=timeout)
    txt = outs_text(outs)
    if any(o.get('output_type') == 'error' for o in outs): raise RuntimeError(txt.strip())
    return txt

## Fast installs with `uv`

Plain `pip install` resolves dependencies on the remote runtime and can overwrite Colab's preinstalled PyTorch with a multi-gigabyte download. Compile a lockfile once locally with `uv pip compile` to skip remote resolution. `uv pip install --system` then installs into Colab's active interpreter in seconds. Pass `no_deps=True` to layer small pure-Python packages over Colab's base image without touching existing dependencies.

In [ ]:
#| export
def compile_reqs(
    pkgs:str|list|Path, # Package spec(s), newline-separated specs, or path to a requirements/pyproject file
    python_version:str='3.13', # Target Python version on the Colab runtime
    extra_args:list=None, # Additional flags passed to `uv pip compile`
)->str:
    "Resolve `pkgs` locally with `uv pip compile` and return pinned requirement lines."
    cmd = ['uv', 'pip', 'compile', '--python-version', python_version, '--quiet', '--no-header', '--no-annotate']
    if extra_args: cmd += list(extra_args)
    if isinstance(pkgs, Path): res = subprocess.run(cmd + [str(pkgs)], capture_output=True, text=True)
    else:
        src = pkgs if isinstance(pkgs, str) else '\n'.join(pkgs)
        res = subprocess.run(cmd + ['-'], input=src + '\n', capture_output=True, text=True)
    if res.returncode: raise RuntimeError(res.stderr.strip() or res.stdout.strip())
    return res.stdout.strip()

Compile a minimal lockfile for `chevron` (a small Mustache template library absent from Colab's base image):

In [ ]:
locked = compile_reqs(['chevron'])
assert locked.startswith('chevron==')
locked

`install` writes any `reqs` input to a temporary file on the runtime and runs `uv pip install --system`. If `uv` is unavailable, it falls back to `pip`.

In [ ]:
#| export
def _read_reqs(reqs:str|Path|None)->str|None:
    if reqs is None: return None
    return reqs.read_text() if isinstance(reqs, Path) else reqs

async def _uv_install(rt:ColabRuntime, pkgs, reqs, target:str=None, no_deps:bool=False, quiet:bool=True, extra_args=None, timeout:float=300)->str:
    pkg_list = [pkgs] if isinstance(pkgs, str) else list(pkgs or [])
    req_txt = _read_reqs(reqs)
    if not pkg_list and not req_txt: raise ValueError('Provide pkgs or reqs')
    common = list(extra_args or []) + (['--no-deps'] if no_deps else []) + (['-q'] if quiet else [])
    tflags = ['--python', target] if target else ['--system']
    code = rf"""
import os, shutil, subprocess, sys, tempfile
pkgs, req_txt, common, tflags, target = {pkg_list!r}, {req_txt!r}, {common!r}, {tflags!r}, {target!r}
req_path = None
try:
    if req_txt is not None:
        fd, req_path = tempfile.mkstemp(prefix='fastcolab-reqs-', suffix='.txt')
        with os.fdopen(fd, 'w') as f: f.write(req_txt + '\n')
        common += ['-r', req_path]
    cmd = (['uv', 'pip', 'install'] + tflags if shutil.which('uv') else [target or sys.executable, '-m', 'pip', 'install']) + common + pkgs
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout, end='')
    if r.stderr: print(r.stderr, end='')
    if r.returncode: raise RuntimeError(f'install failed ({{r.returncode}})')
finally:
    if req_path and os.path.exists(req_path): os.remove(req_path)
"""
    outs = await rt.exec(code, timeout=timeout)
    if any(o.get('output_type') == 'error' for o in outs): raise RuntimeError(outs_text(outs).strip())
    return outs_text(outs)

@patch
async def install(
    self:ColabRuntime,
    pkgs:str|list=None, # Package spec or list of specs to install
    reqs:str|Path=None, # Pinned requirements text or local requirements file path
    no_deps:bool=False, # Pass `--no-deps` to avoid replacing preinstalled packages like PyTorch?
    quiet:bool=True, # Pass `-q` to keep output compact?
    extra_args:list=None, # Additional flags passed to `uv pip install`
    timeout:float=300, # Command timeout in seconds
)->str:
    "Install packages or locked requirements into Colab's system Python using `uv`."
    return await _uv_install(self, pkgs, reqs, no_deps=no_deps, quiet=quiet, extra_args=extra_args, timeout=timeout)

Install the compiled lockfile onto `rt` and verify `chevron` imports immediately inside the remote kernel:

In [ ]:
t0 = time.monotonic()
await rt.install(reqs=locked)
dt = time.monotonic() - t0
rendered = outs_text(await rt.exec("import chevron; chevron.render('Hello {{name}}!', {'name': 'Colab'})")).strip()
assert 'Hello Colab!' in rendered
f'{rendered} (installed in {dt:.1f}s)'

Pass `no_deps=True` to install a small pure-Python package without letting the resolver upgrade anything else:

In [ ]:
await rt.install('chevron==0.14.0', no_deps=True)
outs_text(await rt.exec("import chevron; chevron.__file__")).strip()

## Archived virtual environments

For larger or version-pinned stacks that clash with Colab's system packages, build an isolated virtual environment once on a Colab runtime (matching Colab's Linux x86_64 and Python 3.13 ABI), archive it, and save the archive to Google Drive or Google Cloud Storage. Future runtimes unpack the archive back to the same path in seconds and run scripts with `{venv_dir}/bin/python` without colliding with system packages.

`create_venv` builds the virtual environment with `uv venv` and installs `pkgs` or `reqs` into it. Set `system_site_packages=True` when you want the isolated environment to inherit Colab's preinstalled PyTorch and CUDA instead of downloading them again.

In [ ]:
#| export
@patch
async def create_venv(
    self:ColabRuntime,
    path:str='/content/env', # Directory for the virtual environment on the runtime
    pkgs:str|list=None, # Package spec or list of specs to install into the venv
    reqs:str|Path=None, # Pinned requirements text or local requirements file path
    system_site_packages:bool=False, # Inherit Colab's preinstalled system packages (e.g. PyTorch/CUDA)?
    no_deps:bool=False, # Pass `--no-deps` when installing into the venv?
    quiet:bool=True, # Pass `-q` to `uv`?
    extra_args:list=None, # Extra flags for `uv pip install`
    timeout:float=600, # Command timeout in seconds
)->str:
    "Create an isolated virtual environment on `rt` with `uv` and install packages into it."
    vcmd = ['uv', 'venv', path, '--clear']
    if system_site_packages: vcmd.append('--system-site-packages')
    if quiet: vcmd.append('-q')
    await _run_cmd(self, vcmd, timeout=timeout)
    if pkgs or reqs:
        await _uv_install(self, pkgs, reqs, target=f'{path}/bin/python', no_deps=no_deps, quiet=quiet, extra_args=extra_args, timeout=timeout)
    return path

`venv_exec` runs Python directly under `{venv_dir}/bin/python` for full interpreter isolation. `activate_venv` prepends the venv's `site-packages` directory onto `sys.path` inside the active Jupyter kernel when you want interactive cells to import from it directly.

In [ ]:
#| export
@patch
async def venv_exec(
    self:ColabRuntime,
    code:str, # Python source to run inside the virtual environment
    venv_dir:str='/content/env', # Path to the virtual environment on the runtime
    timeout:float=None, # Execution timeout in seconds
)->str:
    "Run Python `code` using `{venv_dir}/bin/python` for complete isolation from Colab's base packages."
    return await _run_cmd(self, [f'{venv_dir}/bin/python', '-c', code], timeout=timeout)

@patch
async def activate_venv(
    self:ColabRuntime,
    venv_dir:str='/content/env', # Path to the virtual environment on the runtime
)->str:
    "Prepend `{venv_dir}`'s `site-packages` onto `sys.path` inside the active Jupyter kernel."
    code = f'''
import glob, sys
sites = sorted(glob.glob({f'{venv_dir}/lib/python*/site-packages'!r}))
if not sites: raise FileNotFoundError('No site-packages found under ' + {venv_dir!r})
if sites[0] not in sys.path: sys.path.insert(0, sites[0])
sites[0]
'''
    outs = await self.exec(code)
    if any(o.get('output_type') == 'error' for o in outs): raise RuntimeError(outs_text(outs).strip())
    return outs_text(outs).strip().strip("'\"")

Create an isolated virtual environment at `/content/demo_env` containing `shortuuid`, then verify it runs inside `venv_exec` while staying invisible to Colab's base Python:

In [ ]:
await rt.create_venv('/content/demo_env', pkgs=['shortuuid'])
iso_out = (await rt.venv_exec("import shortuuid; print(len(shortuuid.uuid()))", '/content/demo_env')).strip()
base_check = outs_text(await rt.exec("try:\n  import shortuuid\n  print('leaked')\nexcept ImportError:\n  print('isolated')")).strip()
test_eq(iso_out, '22')
test_eq(base_check, 'isolated')
dict(venv_uuid_len=iso_out, base_python=base_check)

`pack_venv` and `unpack_venv` archive and restore a virtual environment with `tar`. They default to gzip (`.tar.gz`) because Colab omits `zstd`. Passing a `.zst` filename switches to `--zstd` when you have installed `zstd` on the runtime. Unpacking back to the original path keeps virtual environment symlinks and script shebangs valid.

In [ ]:
#| export
def _tar_flags(archive:str)->list: return ['--zstd'] if str(archive).endswith('.zst') else ['-z']

@patch
async def pack_venv(
    self:ColabRuntime,
    venv_dir:str='/content/env', # Virtual environment directory on the runtime
    archive:str='/content/env.tar.gz', # Output archive path on the runtime (`.tar.gz` or `.tar.zst`)
    timeout:float=600, # Command timeout in seconds
)->str:
    "Compress `venv_dir` into `archive` on the runtime."
    await _run_cmd(self, ['tar', *_tar_flags(archive), '-cf', archive, '-C', venv_dir, '.'], timeout=timeout)
    return archive

@patch
async def unpack_venv(
    self:ColabRuntime,
    archive:str='/content/env.tar.gz', # Archive path on the runtime (`.tar.gz` or `.tar.zst`)
    dest:str='/content/env', # Destination directory (match the original `venv_dir` path)
    timeout:float=600, # Command timeout in seconds
)->str:
    "Extract `archive` into `dest`, replacing any existing directory at `dest`."
    await _run_cmd(self, ['rm', '-rf', dest], timeout=timeout)
    await _run_cmd(self, ['mkdir', '-p', dest], timeout=timeout)
    await _run_cmd(self, ['tar', *_tar_flags(archive), '-xf', archive, '-C', dest], timeout=timeout)
    return dest

Pack `/content/demo_env` locally on the runtime and check the compressed size:

In [ ]:
await rt.pack_venv('/content/demo_env', '/content/demo_env.tar.gz')
sz = outs_text(await rt.exec("import os; f'{os.path.getsize(\"/content/demo_env.tar.gz\")/1024:.1f} KB'")).strip()
await _run_cmd(rt, ['rm', '-f', '/content/demo_env.tar.gz'])
sz

`save_venv` and `load_venv` wrap `pack_venv` and `unpack_venv` around `to_drive` and `from_drive` for one-call Google Drive persistence. For multi-gigabyte environments, use `gsutil cp` with a Google Cloud Storage bucket instead. Colab includes `gsutil`, and transfers stay inside Google's network.

In [ ]:
#| export
@patch
async def save_venv(
    self:ColabRuntime,
    folder:str, # Destination Google Drive folder name, path, or folder ID
    venv_dir:str='/content/env', # Virtual environment directory to archive
    name:str='env.tar.gz', # Archive filename stored in Drive
    create_folder:bool=True, # Create the Drive folder if missing?
    timeout:float=600, # Pack timeout in seconds
)->dict:
    "Pack `venv_dir` and upload the archive to Google Drive `folder`."
    tmp = f'/content/_fastcolab_{name}'
    try:
        await self.pack_venv(venv_dir, tmp, timeout=timeout)
        return await self.to_drive(tmp, folder, name=name, create_folder=create_folder)
    finally: await _run_cmd(self, ['rm', '-f', tmp])

@patch
async def load_venv(
    self:ColabRuntime,
    folder:str, # Source Google Drive folder name, path, or folder ID
    name:str='env.tar.gz', # Archive filename in Drive
    dest:str='/content/env', # Destination virtual environment directory on the runtime
    timeout:float=600, # Unpack timeout in seconds
)->str:
    "Download `name` from Google Drive `folder` and extract it into `dest`."
    stage = 'content/_fastcolab_stage'
    tmp = f'/{stage}/{name}'
    try:
        pulled = await self.from_drive(folder, remote_dir=stage, names=[name])
        if not pulled: raise FileNotFoundError(f'{name!r} not found in Drive folder {folder!r}')
        return await self.unpack_venv(tmp, dest, timeout=timeout)
    finally: await _run_cmd(self, ['rm', '-rf', f'/{stage}'])

Save `/content/demo_env` to a temporary Google Drive folder, then delete `/content/demo_env` on the runtime. Restore it from Drive and test both `venv_exec` and live kernel imports via `activate_venv`.

In [ ]:
env_folder = f'fastcolab-env-demo-{uuid.uuid4().hex[:8]}'
saved_env = await rt.save_venv(env_folder, '/content/demo_env', name='demo_env.tar.gz')
saved_env

In [ ]:
await _run_cmd(rt, ['rm', '-rf', '/content/demo_env'])
t0 = time.monotonic()
await rt.load_venv(env_folder, name='demo_env.tar.gz', dest='/content/demo_env')
restore_s = time.monotonic() - t0
restored_out = (await rt.venv_exec("import shortuuid; print(len(shortuuid.uuid()))", '/content/demo_env')).strip()
test_eq(restored_out, '22')
site_dir = await rt.activate_venv('/content/demo_env')
kernel_out = outs_text(await rt.exec("import shortuuid; len(shortuuid.uuid())")).strip()
test_eq(kernel_out, '22')
dict(restore_seconds=round(restore_s, 2), site_dir=site_dir, kernel_uuid_len=kernel_out)

Delete the temporary Drive folder so we leave Drive tidy:

In [ ]:
env_fid = await _drive_folder_id(c.drive, env_folder, create=False)
await c.drive.files.delete(file_id=env_fid)

## Reusing a warm runtime

Deleting a runtime wipes every installed package and everything under `/content`. During rapid iteration, restart the Jupyter kernel (or open a fresh one on the same runtime) instead. Restarting clears Python memory, variables, and GPU VRAM in seconds. Installed packages, unpacked virtual environments, and `/content` files stay warm for up to 24 hours.

In [ ]:
#| export
@patch
async def restart(
    self:ColabRuntime,
    timeout:float=60, # Timeout in seconds waiting for the restarted kernel
)->None:
    "Restart `rt`'s default Jupyter kernel without destroying the runtime or `/content`."
    if self._kc is None or not self._kc.channels_running:
        self._kc = await self.kernel(timeout=timeout)
        return
    await self._kc.restart_kernel()
    await self._kc.wait_for_ready(timeout=timeout)

@patch
async def fresh_kernel(
    self:ColabRuntime,
    timeout:float=60, # Connection timeout in seconds
):
    "Shut down `rt`'s current default kernel and open a brand-new kernel on the same runtime."
    if self._kc is not None:
        try: await self._kc.shutdown_kernel()
        except Exception: pass
        self._kc = None
    self._kc = await self.kernel(timeout=timeout)
    return self._kc

Set a variable in memory and write a file to `/content`, then call `rt.restart()`. The in-memory variable disappears while `/content/warm.txt`, `chevron`, and `/content/demo_env` remain on disk.

In [ ]:
await rt.exec("marker = 'before-restart'; open('/content/warm.txt', 'w').write('persisted')")
old_kid = rt._kc.kernel_id
t0 = time.monotonic()
await rt.restart()
restart_s = time.monotonic() - t0
check = outs_text(await rt.exec('''
import chevron
try:
    marker
    var_state = 'still-set'
except NameError:
    var_state = 'cleared'
disk_state = open('/content/warm.txt').read()
print(f'{var_state}|{disk_state}|{chevron.render("{{x}}", {"x": "ok"})}')
''')).strip()
test_eq(check, 'cleared|persisted|ok')
dict(restart_seconds=round(restart_s, 2), same_kernel_id=(rt._kc.kernel_id == old_kid), check=check)

`fresh_kernel` shuts down the current kernel and opens a new kernel ID on the same warm runtime:

In [ ]:
kc2 = await rt.fresh_kernel()
assert kc2.kernel_id != old_kid
warm_after = outs_text(await rt.exec("open('/content/warm.txt').read()")).strip()
assert 'persisted' in warm_after
dict(old_kernel=old_kid, new_kernel=kc2.kernel_id, disk_file=warm_after)

## Cleaning up

Idle runtimes continue burning compute units. Delete the runtime once work ends and check that none remain active.

In [ ]:
await rt.stop()
remaining = await c.runtimes()
test_eq(remaining, [])
remaining

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()